In [6]:
# magic commands to reload custom modules
%load_ext autoreload
%autoreload 2

# magic for matplotlib
%matplotlib notebook

from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import uncertainties.unumpy as unp
import uncertainties as unc
from uncertainties import ufloat, ufloat_fromstr
from mapp_tricks.peakfit import PeakFitter
from mapp_tricks.plotting import apply_my_plotly_style, save_for_presi
from mapp_tricks.utils import parse_csv, store_csv
from scipy.optimize import curve_fit

# old calibration
from mapp_tricks.spectrometer_calibrations import HPGeCalibration as OldHPGeCalibration
from mapp_tricks.spectrometer_calibrations.HPGe.hpge_calib import efficiency_model as old_efficiency_model
from mapp_tricks.spectrometer_calibrations.HPGe.hpge_calib import get_error_vector as old_get_error_vector


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Calibration using source 081025-1979021 - 19.12.2025

In [7]:
calib_source_data = pd.read_csv(('../calibration-data/raw-data/calib-2026/calibration_source_gamma_peaks.csv'))
# convert reference_date to datetime: 2025-11-11 12:00:00
datetime_format = "%Y-%m-%d %H:%M:%S"
calib_source_data['reference_date'] = pd.to_datetime(
    calib_source_data['reference_date'], 
    format=datetime_format
)
# localize reference_date to Europe/Prague timezone since the data was collected in Prague, Czech Republic
calib_source_data['reference_date'] = calib_source_data['reference_date'].dt.tz_localize('Europe/Prague')
# sort by peak_energy_keV
calib_source_data = calib_source_data.sort_values(by='peak_energy_keV').reset_index(drop=True)
calib_source_data.head(n=len(calib_source_data))

,radionuclide,half_life_value,half_life_uncertainty,half_life_unit,half_life_s,half_life_uncertainty_s,peak_energy_keV,peak_energy_uncertainty_keV,roi_lower_keV,roi_upper_keV,intensity,intensity_uncertainty,reference_activity_kBq,reference_activity_uncertainty_kBq,reference_date,notes
0,Am-241,432.6000,0.6000,years,1.365182e+10,18934560.00,59.54090,0.00010,55.2064,64.5224,0.359000,0.004000,4.398,0.031,2025-11-11 12:00:00+01:00,good according to knoll
1,Cd-109,461.9000,0.4000,days,3.990816e+07,34560.00,88.03360,0.00100,84.9824,92.0545,0.036440,0.001600,16.690,0.200,2025-11-11 12:00:00+01:00,only one gamma peak - good
2,Co-57,271.8100,0.0400,days,2.348438e+07,3456.00,122.06065,0.00012,118.1480,126.1960,0.856000,0.001700,1.849,0.013,2025-11-11 12:00:00+01:00,good according to knoll
3,Ce-139,137.6410,0.0200,days,1.189218e+07,1728.00,165.85750,0.00110,162.0440,170.0910,0.799000,0.000500,1.757,0.014,2025-11-11 12:00:00+01:00,only one gamma peak - good
4,Cr-51,27.7040,0.0040,days,2.393626e+06,345.60,320.08240,0.00040,315.6780,323.7260,0.099100,0.000100,21.280,0.150,2025-11-11 12:00:00+01:00,only one gamma peak - good
5,Sr-85,64.8500,0.0070,days,5.603040e+06,604.80,514.00480,0.00220,511.0920,518.9320,0.960000,0.001000,3.968,0.056,2025-11-11 12:00:00+01:00,good according to knoll
6,Cs-137,30.0180,0.0220,years,9.472960e+08,694267.20,661.65700,0.00300,658.3080,667.0870,0.851000,0.002000,2.144,0.017,2025-11-11 12:00:00+01:00,only one gamma peak - good
7,Co-60,5.2711,0.0008,years,1.663433e+08,25246.08,1173.22800,0.00300,1168.7200,1179.2000,0.998500,0.000300,2.668,0.037,2025-11-11 12:00:00+01:00,good according to knoll
8,Co-60,5.2711,0.0008,years,1.663433e+08,25246.08,1332.49200,0.00400,1328.2000,1338.6900,0.999826,0.000006,2.668,0.037,2025-11-11 12:00:00+01:00,good according to knoll
9,Y-88,106.6300,0.0500,days,9.212832e+06,4320.00,1836.06300,0.01200,1828.6000,1847.6800,0.992000,0.003000,6.279,0.057,2025-11-11 12:00:00+01:00,good according to knoll


In [8]:
# efficiency model: sum of (log(E)/E)**n terms up to n=5
def efficiency_model(E, a0, a1, a2, a3, a4, a5):
    h = np.log(E)
    return (1/E) * (
            a0 * 1 +
            a1 * h +
            a2 * h**2 +
            a3 * h**3 +
            a4 * h**4 +
            a5 * h**5
        )
# error vector for the efficiency model, used to plot the error of the fit
def get_error_vector(x, cov_beta):
    """
    Build the error vector for the efficiency model.
    x: energy values [keV]
    cov_beta: covariance matrix of the fit parameters
    """
    sigmas = []
    for x_i in x:
        A = np.zeros((6, 1))
        A[0, 0] = 1 / x_i                 # derivative with respect to a_0
        A[1, 0] = np.log(x_i) / x_i       # derivative with respect to a_1
        A[2, 0] = (np.log(x_i)**2) / x_i  # derivative with respect to a_2
        A[3, 0] = (np.log(x_i)**3) / x_i  # derivative with respect to a_3
        A[4, 0] = (np.log(x_i)**4) / x_i  # derivative with respect to a_4
        A[5, 0] = (np.log(x_i)**5) / x_i  # derivative with respect to a_5
        sigma = np.sqrt(np.diag(A.T @ cov_beta @ A))
        sigmas.append(sigma[0])  # take the first element since J is 6x1
    return np.array(sigmas)

In [ ]:

levels = np.linspace(1, 10, 10, dtype=int)
with_aluminum_foil = False
raw_data_dir = '../calibration-data/raw-data/calib-2026/default'
out_dir = '../calibration-data/processed-data/calib-2026/default'
results = []

compare_to_old_calibration = True

for level in levels:
    print(f'Processing calibration level: level-{level}')

    # and take roi_lower_keV and roi_upper_keV around each peak as list of tuples
    energy_ranges = list(zip(calib_source_data['roi_lower_keV'], calib_source_data['roi_upper_keV']))

    level_string = f'level-{level}'

    fitter = PeakFitter()
    res = fitter.process_file_multiple_peaks(
        filepath=f'{raw_data_dir}/{level_string}.CNF',
        energy_ranges=energy_ranges,
        output_dir=f'{out_dir}/plots/{level_string}',
    )

    # since all the peaks are from the same spectra, we can extract time information from any fit result
    res_pf_list = [r[1] for r in res]
    first_spectra = res_pf_list[0]
    start_time = first_spectra.start_time
    real_time = first_spectra.real_time
    live_time = first_spectra.live_time

    # calculate reference activities at measurement start time
    data = calib_source_data.copy()

    # calculate peak activities by scaling reference activity with intensity
    ref_activities_Bq = unp.uarray(data.reference_activity_kBq, data.reference_activity_uncertainty_kBq) * 1e3  # Bq
    ref_peak_activities_Bq = ref_activities_Bq * unp.uarray(data.intensity, data.intensity_uncertainty) # Bq

    decay_time_s = start_time - data.reference_date
    # again, all peaks are from the same spectra
    decay_time_s = decay_time_s[0].total_seconds()

    half_lifes_s = unp.uarray(data.half_life_s, data.half_life_uncertainty_s)
    lambdas_s_inv = np.log(2) / half_lifes_s

    # reference peak activities at measurement start
    ref_peak_activities_at_measurement_start_Bq = ref_peak_activities_Bq * unp.exp(-lambdas_s_inv * decay_time_s)


    measured_peak_activities: list[ufloat] = []
    for peak_fit_result, decay_constant in zip(res_pf_list, lambdas_s_inv):
        correction = (decay_constant * real_time) / (1 - unp.exp(-decay_constant * real_time)) # correction factor for decay during measurement (>1)
        mpa = (peak_fit_result.area / live_time) * correction
        # print(f'peak {peak_fit_result.centroid}, activity {mpa}, correction {correction}')
        measured_peak_activities.append(mpa)

    data['measured_peak_activity_Bq'] = measured_peak_activities
    data['efficiency'] = data.measured_peak_activity_Bq / ref_peak_activities_at_measurement_start_Bq

    # fit efficiency model to data
    popt, pcov = curve_fit(
        efficiency_model,
        unp.nominal_values(data.peak_energy_keV),
        unp.nominal_values(data.efficiency),
        sigma=unp.std_devs(data.efficiency),
        absolute_sigma=True,
    )

    params_unc = unc.correlated_values(popt, pcov)


    fig = go.Figure()

    # fitted curve data
    energy_fit = np.linspace(min(data.peak_energy_keV)*1.0, max(data.peak_energy_keV)*1.1, 1000)
    efficiency_fit = efficiency_model(energy_fit, *popt)
    errors_fit = get_error_vector(energy_fit, pcov)

    # add error band
    fig.add_trace(go.Scatter(
        x=np.concatenate([energy_fit, energy_fit[::-1]]),
        y=np.concatenate([efficiency_fit - errors_fit, (efficiency_fit + errors_fit)[::-1]]),
        fill='toself',
        fillcolor='rgba(255,0,0,0.5)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        showlegend=False,
        name='Fit uncertainty'
    ))
    # add fitted curve 
    fig.add_trace(go.Scatter(
        x=energy_fit,
        y=efficiency_fit,
        mode='lines',
        line=dict(color='rgba(255,0,0,1)'),
        name='Fitted efficiency curve'
    ))

    # add data
    fig.add_trace(go.Scatter(
        x=unp.nominal_values(data.peak_energy_keV),
        error_x=dict(
            type='data',
            array=unp.std_devs(data.peak_energy_keV),
            visible=True
        ),
        y=unp.nominal_values(data.efficiency),
        error_y=dict(
            type='data',
            array=unp.std_devs(data.efficiency),
            visible=True
        ),
        marker=dict(
            color='rgba(50,50,50,1)',
        ),
        mode='markers',
        name='Efficiency measured'
    ))

    fig.update_layout(
        title=f'HPGe Efficiency Calibration - {level_string}',
        xaxis_title='Energy [keV]',
        yaxis_title='Efficiency',
        )

    # just for comparison compare to old calibration
    if compare_to_old_calibration:
        old_calib = OldHPGeCalibration(level=level, with_aluminum_foil=with_aluminum_foil)

        Ey_old = unp.nominal_values(old_calib.data['energy'])
        eff_old_vals = unp.nominal_values(old_calib.data['efficiency'])
        eff_old_err_vals = unp.std_devs(old_calib.data['efficiency'])

        eff_old = old_efficiency_model(energy_fit, *old_calib.fit_data.fit_params)
        eff_old_errors = old_get_error_vector(energy_fit, old_calib.fit_data.fit_covariance)

        fig.add_trace(go.Scatter(
            x=np.concatenate([energy_fit, energy_fit[::-1]]),
            y=np.concatenate([eff_old - eff_old_errors, (eff_old + eff_old_errors)[::-1]]),
            fill='toself',
            fillcolor='rgba(0,0,255,0.5)',
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo="skip",
            showlegend=False,
            name='Old fit uncertainty'
        ))
        fig.add_trace(go.Scatter(
            x=energy_fit,
            y=eff_old,
            mode='lines',
            line=dict(color='rgba(0,0,255,1)'),
            name='Old fitted efficiency curve'
        ))
        
        fig.add_trace(go.Scatter(
            x=Ey_old,
            y=eff_old_vals,
            error_y=dict(
                type='data',
                array=eff_old_err_vals,
                visible=True
            ),
            marker=dict(
                color='rgba(0,0,255,1)',
            ),
            mode='markers',
            name='Old calibration data'
        ))




    # fig.write_image(f'{out_dir}/{level_string}/efficiency_calibration_plot.pdf')
    fig = apply_my_plotly_style(fig)
    save_for_presi(fig, f'{out_dir}/plots/{level_string}/efficiency_calibration_plot.pdf')

    # store fit results in a dict
    results.append({
        'level': level,
        'with_aluminum_foil': with_aluminum_foil,
        'start_time': start_time,
        'real_time': real_time,
        'live_time': live_time,
        'a0': params_unc[0],
        'a1': params_unc[1],
        'a2': params_unc[2],
        'a3': params_unc[3],
        'a4': params_unc[4],
        'a5': params_unc[5],
    })

    fig.show()

# save results to CSV
results_df = pd.DataFrame(results)
store_csv(results_df, f'{out_dir}/efficiency_calibration_results.csv')



Processing calibration level: level-1


peakfit - processing energy ranges:   0%|          | 0/10 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>